In [1]:
import os
print(os.listdir())

['.copilot', '.idlerc', '.ipynb_checkpoints', '.ipython', '.jupyter', '.ms-ad', '.vscode', '.vscode-shared', 'ADAP.ipynb', 'ADAP2.ipynb', 'AP3.ipynb', 'AppData', 'Application Data', 'ASP.ipynb', 'ASP.py', 'Congressional Voting.ipynb', 'Contacts', 'Cookies', 'data clng.ipynb', 'decisions.db', 'Decisiontree.ipynb', 'Documents', 'Downloads', 'DT.ipynb', 'F .ipynb', 'F SP .ipynb', 'Favorites', 'feedback_data.csv', 'iris_model.pkl', 'K-means.ipynb', 'Links', 'Local Settings', 'Microsoft', 'Microsoft VS Code', 'ml_engine', 'Music', 'My Documents', 'N,S,P.ipynb', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{af6197b5-2fcd-11f0-9681-eada096b09d8}.TM.blf', 'NTUSER.DAT{af6197b5-2fcd-11f0-9681-eada096b09d8}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{af6197b5-2fcd-11f0-9681-eada096b09d8}.TMContainer00000000000000000002.regtrans-ms', 'ntuser.ini', 'OneDrive', 'plot.png', 'Practical1.ipynb', 'practical2.ipynb', 'PrintHood', 'Programs', 'PTime.ipynb', 'Rece

In [2]:
import os

if os.path.exists("xgboost_delay_model.pkl"):
    os.remove("xgboost_delay_model.pkl")
    print("✅ Old model deleted")

✅ Old model deleted


In [8]:
import joblib
model = joblib.load(r"C:\Users\GEETHA\ml_engine\xgboost_delay_model.pkl")

In [10]:
import sqlite3
import pandas as pd
import numpy as np
import joblib
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Load trained model from Week 1
model = joblib.load(r"C:\Users\GEETHA\ml_engine\xgboost_delay_model.pkl")

In [11]:
conn = sqlite3.connect("supply_chain.db")
cursor = conn.cursor()

# Table to store prediction vs actual
cursor.execute("""
CREATE TABLE IF NOT EXISTS feedback (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    distance REAL,
    weather REAL,
    traffic REAL,
    predicted_delay REAL,
    actual_delay REAL,
    error REAL
)
""")

conn.commit()

In [12]:
def predict_delay(distance, weather, traffic):
    X = np.array([[distance, weather, traffic]])
    return model.predict(X)[0]

In [13]:
def load_or_initialize_model():
    if os.path.exists("xgboost_delay_model.pkl"):
        print("📦 Loading existing model...")
        return joblib.load("xgboost_delay_model.pkl")
    else:
        print("⚠️ Creating new model with 3 features...")

        model = XGBRegressor()

        # Dummy training (3 features)
        X_dummy = np.array([
            [100, 20, 30],
            [200, 40, 50],
            [300, 60, 70],
            [400, 80, 90]
        ])
        y_dummy = np.array([30, 60, 90, 120])

        model.fit(X_dummy, y_dummy)

        joblib.dump(model, "xgboost_delay_model.pkl")
        return model

In [14]:
def store_feedback(distance, weather, traffic, actual_delay):

    # Convert inputs to float (important)
    distance = float(distance)
    weather = float(weather)
    traffic = float(traffic)
    actual_delay = float(actual_delay)

    predicted = predict_delay(distance, weather, traffic)
    error = abs(predicted - actual_delay)

    cursor.execute("""
    INSERT INTO feedback (distance, weather, traffic, predicted_delay, actual_delay, error)
    VALUES (?, ?, ?, ?, ?, ?)
    """, (distance, weather, traffic, predicted, actual_delay, error))

    conn.commit()

    print("✅ Feedback stored!")
    print(f"Predicted: {predicted:.2f}, Actual: {actual_delay}, Error: {error:.2f}")

    return error

In [15]:
def check_retraining(threshold=10, min_records=5):
    
    df = pd.read_sql_query("SELECT * FROM feedback", conn)

    if len(df) < min_records:
        print("ℹ️ Not enough data for retraining")
        return False

    avg_error = df["error"].mean()
    print(f"📊 Average Error: {avg_error:.2f}")

    if avg_error > threshold:
        print("⚠️ Triggering retraining...")
        return True
    else:
        print("✅ Model is performing well")
        return False

In [16]:
def retrain_model():
    
    df = pd.read_sql_query("SELECT * FROM feedback", conn)

    # 🔥 Convert all columns to numeric (VERY IMPORTANT)
    df["distance"] = pd.to_numeric(df["distance"], errors='coerce')
    df["weather"] = pd.to_numeric(df["weather"], errors='coerce')
    df["traffic"] = pd.to_numeric(df["traffic"], errors='coerce')
    df["actual_delay"] = pd.to_numeric(df["actual_delay"], errors='coerce')

    # Drop invalid rows
    df = df.dropna()

    X = df[["distance", "weather", "traffic"]]
    y = df["actual_delay"]

    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

    from xgboost import XGBRegressor
    new_model = XGBRegressor()
    new_model.fit(X_train, y_train)

    from sklearn.metrics import mean_squared_error
    import numpy as np
    preds = new_model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    print(f"🔁 New Model RMSE: {rmse:.2f}")

    import joblib
    joblib.dump(new_model, "xgboost_delay_model.pkl")

    print("✅ Model retrained & saved!")

    return new_model

In [17]:
def continuous_learning_pipeline(f1, f2, f3, actual_delay):
    print("\n🚀 Running Closed Loop Pipeline...")

    # ✅ Load model safely
    model = load_or_initialize_model()

    import numpy as np
    input_data = np.array([f1, f2, f3], dtype=float)

    prediction = model.predict([input_data])[0]

    error = abs(prediction - actual_delay)

    print(f"✅ Feedback stored!")
    print(f"Predicted: {prediction:.2f}, Actual: {actual_delay}, Error: {error:.2f}")

In [18]:
import os
import pandas as pd
from xgboost import XGBRegressor
import joblib

print("\n🚀 Running Closed Loop Pipeline...")

# Check if feedback file exists
if not os.path.exists("feedback_data.csv"):
    print("ℹ️ No feedback data yet. Skipping retraining.")
else:
    df = pd.read_csv("feedback_data.csv")

    # Check minimum data requirement
    if len(df) < 5:
        print("ℹ️ Not enough data for retraining")
    else:
        # Clean data (important!)
        def clean_dataframe(df):
            for col in df.columns:
                df[col] = df[col].apply(
                    lambda x: float(x.decode()) if isinstance(x, bytes) else float(x)
                )
            return df

        df = clean_dataframe(df)

        # Split features & target
        X = df.drop(columns=["actual"])
        y = df["actual"]

        # Train model
        model = XGBRegressor()
        model.fit(X, y)

        # Save model
        joblib.dump(model, "xgboost_delay_model.pkl")

        print("🔄 Model retrained successfully!")

print("\n✅ Pipeline completed!")


🚀 Running Closed Loop Pipeline...
🔄 Model retrained successfully!

✅ Pipeline completed!


In [19]:
# Simulate real-world inputs
continuous_learning_pipeline(400, 60, 70, actual_delay=85)
continuous_learning_pipeline(300, 40, 50, actual_delay=60)
continuous_learning_pipeline(500, 80, 90, actual_delay=120)
continuous_learning_pipeline(200, 30, 40, actual_delay=45)
continuous_learning_pipeline(350, 55, 65, actual_delay=75)
continuous_learning_pipeline(450, 70, 85, actual_delay=110)


🚀 Running Closed Loop Pipeline...
📦 Loading existing model...
✅ Feedback stored!
Predicted: 85.00, Actual: 85, Error: 0.00

🚀 Running Closed Loop Pipeline...
📦 Loading existing model...
✅ Feedback stored!
Predicted: 60.00, Actual: 60, Error: 0.00

🚀 Running Closed Loop Pipeline...
📦 Loading existing model...
✅ Feedback stored!
Predicted: 120.00, Actual: 120, Error: 0.00

🚀 Running Closed Loop Pipeline...
📦 Loading existing model...
✅ Feedback stored!
Predicted: 45.00, Actual: 45, Error: 0.00

🚀 Running Closed Loop Pipeline...
📦 Loading existing model...
✅ Feedback stored!
Predicted: 75.00, Actual: 75, Error: 0.00

🚀 Running Closed Loop Pipeline...
📦 Loading existing model...
✅ Feedback stored!
Predicted: 85.00, Actual: 110, Error: 25.00


In [20]:
import pandas as pd
import os

def store_feedback(f1, f2, f3, actual, predicted):
    new_data = pd.DataFrame([[f1, f2, f3, actual]], 
                            columns=["f1", "f2", "f3", "actual"])

    if os.path.exists("feedback_data.csv"):
        new_data.to_csv("feedback_data.csv", mode='a', header=False, index=False)
    else:
        new_data.to_csv("feedback_data.csv", index=False)

In [21]:
def retrain_model_if_needed():
    import pandas as pd
    import os
    from xgboost import XGBRegressor
    import joblib

    if not os.path.exists("feedback_data.csv"):
        print("ℹ️ No feedback data yet")
        return

    df = pd.read_csv("feedback_data.csv")

    if len(df) < 5:
        print("ℹ️ Not enough data for retraining")
        return

    choice = input("⚠️ Retrain model? (yes/no): ")

    if choice.lower() != "yes":
        print("⏭️ Retraining skipped by analyst")
        return

    print("🔄 Retraining model...")

    X = df[["feature1", "feature2"]]
    y = df["actual"]

    model = XGBRegressor()
    model.fit(X, y)

    joblib.dump(model, "xgboost_delay_model.pkl")

    print("✅ Model retrained successfully!")

In [22]:
def retrain_model_if_needed():
    if not os.path.exists("feedback_data.csv"):
        print("ℹ️ Not enough data for retraining")
        return

    df = pd.read_csv("feedback_data.csv")

    if len(df) < 5:
        print("ℹ️ Not enough data for retraining")
        return

    choice = input("⚠️ Retrain model? (yes/no): ")

    if choice.lower() == "yes":
        print("🔄 Retraining model...")

        # ✅ FIX IS HERE
        X = df[['f1', 'f2', 'f3']]
        y = df['actual']

        from xgboost import XGBRegressor
        model = XGBRegressor()
        model.fit(X, y)

        import joblib
        joblib.dump(model, "xgboost_delay_model.pkl")

        print("✅ Model retrained successfully!")
    else:
        print("⏭️ Skipping retraining")

In [23]:
def continuous_learning_pipeline(f1, f2, f3, actual_delay):
    print("\n🚀 Running Closed Loop Pipeline...")

    model = load_or_initialize_model()

    import numpy as np
    input_data = np.array([f1, f2, f3], dtype=float)

    prediction = model.predict([input_data])[0]
    error = abs(prediction - actual_delay)

    print(f"✅ Feedback stored!")
    print(f"Predicted: {prediction:.2f}, Actual: {actual_delay}, Error: {error:.2f}")

    # ✅ Store feedback
    store_feedback(f1, f2, f3, actual_delay, prediction)

    # ✅ Retrain
    retrain_model_if_needed()

In [ ]:
continuous_learning_pipeline(400, 60, 70, actual_delay=85)
continuous_learning_pipeline(300, 40, 50, actual_delay=60)
continuous_learning_pipeline(500, 80, 90, actual_delay=120)
continuous_learning_pipeline(200, 30, 40, actual_delay=45)
continuous_learning_pipeline(350, 55, 65, actual_delay=75)
continuous_learning_pipeline(450, 70, 85, actual_delay=110)


🚀 Running Closed Loop Pipeline...
📦 Loading existing model...
✅ Feedback stored!
Predicted: 85.00, Actual: 85, Error: 0.00


⚠️ Retrain model? (yes/no):  yes


🔄 Retraining model...
✅ Model retrained successfully!

🚀 Running Closed Loop Pipeline...
📦 Loading existing model...
✅ Feedback stored!
Predicted: 60.00, Actual: 60, Error: 0.00


⚠️ Retrain model? (yes/no):  yes


🔄 Retraining model...
✅ Model retrained successfully!

🚀 Running Closed Loop Pipeline...
📦 Loading existing model...
✅ Feedback stored!
Predicted: 120.00, Actual: 120, Error: 0.00


⚠️ Retrain model? (yes/no):  yes


🔄 Retraining model...
✅ Model retrained successfully!

🚀 Running Closed Loop Pipeline...
📦 Loading existing model...
✅ Feedback stored!
Predicted: 45.00, Actual: 45, Error: 0.00


In [ ]:
import pandas as pd

df = pd.read_csv("feedback_data.csv")
print(df.columns)